Tabela Fato de Avaliações — Camada Ouro

Objetivo

Construir a tabela fato de avaliações da camada Ouro a partir da tabela
harmonizada da camada Prata e das dimensões construídas no notebook 09.

Granularidade

Cada registro da tabela fato representa uma avaliação individual.

Chave da avaliação:

- `id_avaliacao_global`

Dimensões relacionadas

- `dim_fonte`
- `dim_filme`
- `dim_data`

Princípios

- preservar os registros da camada Prata;
- reutilizar identificadores já existentes;
- não eliminar avaliações sem filme ou data;
- não normalizar `nota_original` sem regra de negócio explícita;
- validar cardinalidade e integridade antes da persistência.

In [0]:
# ============================================================
# Configuração, leitura e validação do contrato
# ============================================================

from pyspark.sql import functions as F

TABELA_PRATA = "mvp3_eng_dados.prata.avaliacoes_harmonizadas"

TABELA_DIM_FONTE = "mvp3_eng_dados.ouro.dim_fonte"
TABELA_DIM_FILME = "mvp3_eng_dados.ouro.dim_filme"
TABELA_DIM_DATA = "mvp3_eng_dados.ouro.dim_data"

TABELA_FATO = "mvp3_eng_dados.ouro.fato_avaliacao"

df_prata = spark.table(TABELA_PRATA)
df_dim_fonte = spark.table(TABELA_DIM_FONTE)
df_dim_filme = spark.table(TABELA_DIM_FILME)
df_dim_data = spark.table(TABELA_DIM_DATA)

colunas_necessarias = {
    "id_avaliacao_global",
    "id_avaliacao",
    "fonte",
    "tipo_avaliador",
    "id_filme",
    "texto_avaliacao",
    "classificacao_original",
    "sentimento",
    "nota_original",
    "data_avaliacao",
    "nome_avaliador",
    "publicacao",
    "eh_top_critic"
}

colunas_ausentes = colunas_necessarias - set(df_prata.columns)

print(f"Tabela de entrada: {TABELA_PRATA}")
print(f"Quantidade de colunas: {len(df_prata.columns)}")
print(f"Colunas ausentes: {sorted(colunas_ausentes)}")

assert len(colunas_ausentes) == 0, (
    f"Contrato de entrada inválido. "
    f"Colunas ausentes: {sorted(colunas_ausentes)}"
)

print("\n✓ Contrato de entrada validado.")

Tabela de entrada: mvp3_eng_dados.prata.avaliacoes_harmonizadas
Quantidade de colunas: 16
Colunas ausentes: []

✓ Contrato de entrada validado.


In [0]:
# ============================================================
# Controle da granularidade de entrada
# ============================================================

total_prata = df_prata.count()

ids_avaliacao_distintos = (
    df_prata
    .select("id_avaliacao_global")
    .distinct()
    .count()
)

duplicidades_entrada = (
    total_prata - ids_avaliacao_distintos
)

print(f"Registros da Prata:          {total_prata:,}")
print(f"IDs globais distintos:       {ids_avaliacao_distintos:,}")
print(f"Duplicidades encontradas:     {duplicidades_entrada:,}")

assert total_prata == ids_avaliacao_distintos, (
    "A granularidade da tabela harmonizada não é 1 avaliação por linha."
)

print("\n✓ Granularidade da entrada validada.")

Registros da Prata:          1,060,128
IDs globais distintos:       1,060,128
Duplicidades encontradas:     0

✓ Granularidade da entrada validada.


In [0]:
# ============================================================
# Relacionamento com as dimensões
# ============================================================

df_fato_base = (
    df_prata.alias("a")

    .join(
        df_dim_fonte.alias("f"),
        F.col("a.fonte") == F.col("f.fonte"),
        "left"
    )

    .join(
        df_dim_filme.alias("m"),
        F.col("a.id_filme") == F.col("m.id_filme"),
        "left"
    )

    .join(
        df_dim_data.alias("d"),
        F.col("a.data_avaliacao") == F.col("d.data_avaliacao"),
        "left"
    )

    .select(
        F.col("a.id_avaliacao_global"),
        F.col("a.id_avaliacao"),

        F.col("f.id_fonte"),

        F.col("a.id_filme"),
        F.col("d.id_data"),

        F.col("a.tipo_avaliador"),
        F.col("a.texto_avaliacao"),
        F.col("a.classificacao_original"),
        F.col("a.sentimento"),
        F.col("a.nota_original"),
        F.col("a.data_avaliacao"),
        F.col("a.nome_avaliador"),
        F.col("a.publicacao"),
        F.col("a.eh_top_critic")
    )
)

print(
    f"Registros após relacionamento com dimensões: "
    f"{df_fato_base.count():,}"
)

Registros após relacionamento com dimensões: 1,060,128


In [0]:
# ============================================================
# Validação da cardinalidade após os relacionamentos
# ============================================================

total_fato_base = df_fato_base.count()

ids_fato_distintos = (
    df_fato_base
    .select("id_avaliacao_global")
    .distinct()
    .count()
)

diferenca_registros = total_fato_base - total_prata

print(f"Registros antes dos JOINs:   {total_prata:,}")
print(f"Registros após os JOINs:     {total_fato_base:,}")
print(f"IDs globais distintos:       {ids_fato_distintos:,}")
print(f"Diferença de registros:      {diferenca_registros:,}")

assert total_fato_base == total_prata, (
    "Os relacionamentos alteraram a quantidade de avaliações."
)

assert ids_fato_distintos == total_prata, (
    "A granularidade da fato foi alterada após os relacionamentos."
)

print("\n✓ Cardinalidade preservada.")

Registros antes dos JOINs:   1,060,128
Registros após os JOINs:     1,060,128
IDs globais distintos:       1,060,128
Diferença de registros:      0

✓ Cardinalidade preservada.


In [0]:
# ============================================================
# Cobertura das chaves dimensionais
# ============================================================

df_cobertura = df_fato_base.agg(

    F.sum(
        F.when(F.col("id_fonte").isNull(), 1).otherwise(0)
    ).alias("sem_id_fonte"),

    F.sum(
        F.when(F.col("id_filme").isNull(), 1).otherwise(0)
    ).alias("sem_id_filme"),

    F.sum(
        F.when(F.col("id_data").isNull(), 1).otherwise(0)
    ).alias("sem_id_data")

)

display(df_cobertura)

sem_id_fonte,sem_id_filme,sem_id_data
0,49712,49582


In [0]:
# ============================================================
# Validação crítica e preparação da fato
# ============================================================

sem_id_fonte = (
    df_fato_base
    .filter(F.col("id_fonte").isNull())
    .count()
)

assert sem_id_fonte == 0, (
    "Existem avaliações sem correspondência na dimensão fonte."
)

df_fato_avaliacao = (
    df_fato_base
    .select(
        "id_avaliacao_global",
        "id_avaliacao",
        "id_fonte",
        "id_filme",
        "id_data",
        "tipo_avaliador",
        "texto_avaliacao",
        "classificacao_original",
        "sentimento",
        "nota_original",
        "data_avaliacao",
        "nome_avaliador",
        "publicacao",
        "eh_top_critic"
    )
)

print(f"Registros preparados: {df_fato_avaliacao.count():,}")
print("✓ Tabela fato preparada para persistência.")

Registros preparados: 1,060,128
✓ Tabela fato preparada para persistência.


In [0]:
# ============================================================
# Persistência da tabela fato
# ============================================================

(
    df_fato_avaliacao
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_FATO)
)

print(f"✓ Tabela persistida: {TABELA_FATO}")

✓ Tabela persistida: mvp3_eng_dados.ouro.fato_avaliacao


In [0]:
# ============================================================
# Validação da tabela fato persistida
# ============================================================

df_fato_persistida = spark.table(TABELA_FATO)

total_persistido = df_fato_persistida.count()

ids_persistidos_distintos = (
    df_fato_persistida
    .select("id_avaliacao_global")
    .distinct()
    .count()
)

print(f"Registros na Prata:          {total_prata:,}")
print(f"Registros persistidos:       {total_persistido:,}")
print(f"IDs globais distintos:       {ids_persistidos_distintos:,}")

assert total_persistido == total_prata, (
    "Quantidade persistida diferente da camada Prata."
)

assert ids_persistidos_distintos == total_persistido, (
    "Existem IDs globais duplicados na tabela fato."
)

print("\n✓ Persistência e granularidade validadas.")

Registros na Prata:          1,060,128
Registros persistidos:       1,060,128
IDs globais distintos:       1,060,128

✓ Persistência e granularidade validadas.


In [0]:
# ============================================================
# Perfil final da tabela fato
# ============================================================

df_resumo_fato = df_fato_persistida.agg(

    F.count("*").alias("registros"),

    F.countDistinct(
        "id_avaliacao_global"
    ).alias("avaliacoes_distintas"),

    F.countDistinct(
        "id_fonte"
    ).alias("fontes"),

    F.countDistinct(
        "id_filme"
    ).alias("filmes"),

    F.countDistinct(
        "id_data"
    ).alias("datas"),

    F.sum(
        F.when(F.col("id_fonte").isNull(), 1).otherwise(0)
    ).alias("sem_fonte"),

    F.sum(
        F.when(F.col("id_filme").isNull(), 1).otherwise(0)
    ).alias("sem_filme"),

    F.sum(
        F.when(F.col("id_data").isNull(), 1).otherwise(0)
    ).alias("sem_data"),

    F.sum(
        F.when(F.col("texto_avaliacao").isNull(), 1).otherwise(0)
    ).alias("sem_texto")
)

display(df_resumo_fato)

registros,avaliacoes_distintas,fontes,filmes,datas,sem_fonte,sem_filme,sem_data,sem_texto
1060128,1060128,2,17706,8015,0,49712,49582,58526


In [0]:
# ============================================================
# Encerramento
# ============================================================

print("=" * 65)
print("10_FATO_AVALIACAO_OURO — CONCLUÍDO")
print("=" * 65)

print(f"\nTabela criada:")
print(f"✓ {TABELA_FATO}")

print(f"\nRegistros:")
print(f"✓ Prata: {total_prata:,}")
print(f"✓ Ouro:  {total_persistido:,}")

print("\nValidações:")
print("✓ granularidade preservada")
print("✓ id_avaliacao_global único")
print("✓ relacionamento com dim_fonte validado")
print("✓ registros sem filme preservados")
print("✓ registros sem data preservados")
print("✓ tabela persistida em Delta")

print(
    "\nPróxima etapa: "
    "11_validacao_qualidade_ouro"
)

10_FATO_AVALIACAO_OURO — CONCLUÍDO

Tabela criada:
✓ mvp3_eng_dados.ouro.fato_avaliacao

Registros:
✓ Prata: 1,060,128
✓ Ouro:  1,060,128

Validações:
✓ granularidade preservada
✓ id_avaliacao_global único
✓ relacionamento com dim_fonte validado
✓ registros sem filme preservados
✓ registros sem data preservados
✓ tabela persistida em Delta

Próxima etapa: 11_validacao_qualidade_ouro
